In [1]:
# Install PySpark
!pip install pyspark==3.5.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 11.3 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488493 sha256=68a801dac2a4f4bff2892b187b0eee7d876be850fcefdb156dc87fe9cfb168e9
  Stored in directory: /root/.cache/pip/wheels/b1/91/5f/283b53010a8016a4ff1c4a1edd99bbe73afacb099645b5471b
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.2
    Uninstalling pyspark-4.0.2:
      Successfully uninstalled pyspark-4.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-conn

In [2]:
# Import SparkSession
from pyspark.sql import SparkSession

# Create Spark Session
spark = SparkSession.builder \
    .appName("Week 5 Assignment") \
    .master("local[*]") \
    .getOrCreate()

In [3]:
# Create Sample DataFrame

data = [
    (1, "West", "Electronics", 500),
    (2, "West", "Clothing", 300),
    (3, "East", "Electronics", 700),
    (4, "West", "Electronics", 900),
]

columns = ["id", "region", "product_category", "sale_amount"]

df_sales = spark.createDataFrame(data, columns)

df_sales.show()

+---+------+----------------+-----------+
| id|region|product_category|sale_amount|
+---+------+----------------+-----------+
|  1|  West|     Electronics|        500|
|  2|  West|        Clothing|        300|
|  3|  East|     Electronics|        700|
|  4|  West|     Electronics|        900|
+---+------+----------------+-----------+



# Q1. What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

Traditional MapReduce has several limitations that make it less efficient for modern big data applications:

1. **Disk-Based Processing**
   - MapReduce writes intermediate results to disk after every stage, which increases execution time.

2. **Slow Performance**
   - Frequent disk I/O operations make MapReduce slower, especially for iterative tasks.

3. **Not Suitable for Iterative Algorithms**
   - Machine learning and graph processing require repeated computations, but MapReduce reloads data from disk each time.

4. **Complex Programming Model**
   - Developers need to write separate Map and Reduce functions, making the code lengthy and difficult to maintain.

5. **High Latency**
   - It is not suitable for real-time or interactive data processing because of slower execution.

### Why Spark is Preferred

- Spark performs **in-memory computation**, reducing disk I/O.
- It provides **faster execution** compared to MapReduce.
- It supports **batch processing, streaming, machine learning, and graph analytics**.
- Spark offers easy-to-use APIs in Python, Java, Scala, and R.
- It is well suited for iterative algorithms and real-time analytics.

# Q2. Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

Apache Spark uses **In-Memory Computing**, which means it stores intermediate data in RAM instead of writing it to disk after every operation.

### How it works:

1. **Data is loaded into memory (RAM)** only once.
2. **Intermediate results are cached in memory**, reducing the need for repeated disk reads and writes.
3. **Iterative algorithms**, such as machine learning and graph processing, can reuse the cached data multiple times.
4. This significantly reduces execution time and improves overall performance.

### Comparison with Disk-Based Systems (MapReduce)

| Spark (In-Memory) | MapReduce (Disk-Based) |
|-------------------|------------------------|
| Stores intermediate data in RAM | Stores intermediate data on disk |
| Faster execution | Slower due to disk I/O |
| Ideal for iterative algorithms | Inefficient for repeated computations |
| Low latency | High latency |

### Conclusion

Spark's in-memory processing minimizes disk access, making machine learning algorithms much faster and more efficient than traditional disk-based systems like MapReduce.

# Q3. Remove Duplicate Rows Based on Specific Columns

In this example, duplicate records are removed based on the combination of
`user_id` and `transaction_date` using the `dropDuplicates()` function.

In [4]:
from pyspark.sql import SparkSession

# Create Spark Session
spark = SparkSession.builder.appName("Q3").getOrCreate()

# Sample Data
data = [
    (101, "2025-06-01", 500),
    (102, "2025-06-02", 700),
    (101, "2025-06-01", 500),   # Duplicate
    (103, "2025-06-03", 900),
    (102, "2025-06-02", 700)    # Duplicate
]

columns = ["user_id", "transaction_date", "amount"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

print("Original Data:")
df.show()

# Remove duplicates based on user_id and transaction_date
df_clean = df.dropDuplicates(["user_id", "transaction_date"])

print("Data After Removing Duplicates:")
df_clean.show()

Original Data:
+-------+----------------+------+
|user_id|transaction_date|amount|
+-------+----------------+------+
|    101|      2025-06-01|   500|
|    102|      2025-06-02|   700|
|    101|      2025-06-01|   500|
|    103|      2025-06-03|   900|
|    102|      2025-06-02|   700|
+-------+----------------+------+

Data After Removing Duplicates:
+-------+----------------+------+
|user_id|transaction_date|amount|
+-------+----------------+------+
|    102|      2025-06-02|   700|
|    101|      2025-06-01|   500|
|    103|      2025-06-03|   900|
+-------+----------------+------+



# Q4. Filter West Region and Calculate Average Sale Amount

This query filters the records where the `region` is **'West'** and then groups
the data by `product_category` to calculate the average `sale_amount`.

In [5]:
# Import required function
from pyspark.sql.functions import avg

# Sample Sales Data
data = [
    ("West", "Electronics", 500),
    ("West", "Electronics", 700),
    ("West", "Clothing", 300),
    ("East", "Electronics", 600),
    ("West", "Clothing", 400),
    ("South", "Furniture", 800)
]

columns = ["region", "product_category", "sale_amount"]

# Create DataFrame
df_sales = spark.createDataFrame(data, columns)

# Display Original Data
print("Original Sales Data:")
df_sales.show()

# Filter for West region and calculate average sale amount
result = df_sales.filter(df_sales.region == "West") \
                 .groupBy("product_category") \
                 .agg(avg("sale_amount").alias("Average_Sale_Amount"))

# Display Result
print("Average Sale Amount for West Region:")
result.show()

Original Sales Data:
+------+----------------+-----------+
|region|product_category|sale_amount|
+------+----------------+-----------+
|  West|     Electronics|        500|
|  West|     Electronics|        700|
|  West|        Clothing|        300|
|  East|     Electronics|        600|
|  West|        Clothing|        400|
| South|       Furniture|        800|
+------+----------------+-----------+

Average Sale Amount for West Region:
+----------------+-------------------+
|product_category|Average_Sale_Amount|
+----------------+-------------------+
|     Electronics|              600.0|
|        Clothing|              350.0|
+----------------+-------------------+



# Q5. Difference Between .na.drop() and .na.fill()

### .na.drop()
- Removes rows that contain null (missing) values.
- It is useful when incomplete records should be discarded.

### .na.fill()
- Replaces null values with a specified value.
- It is useful when you want to keep all records by filling missing values.

### Difference

| .na.drop() | .na.fill() |
|------------|------------|
| Removes rows with null values | Replaces null values with a specified value |
| Reduces the number of records | Keeps all records intact |
| Used when missing data is not useful | Used when missing data can be replaced |

In [6]:
# Sample Data
data = [
    (1, "Active"),
    (2, None),
    (3, "Inactive"),
    (4, None)
]

columns = ["id", "status"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

print("Original Data:")
df.show()

# Fill null values in status column with 'Unknown'
df_filled = df.na.fill({"status": "Unknown"})

print("After Filling Null Values:")
df_filled.show()

Original Data:
+---+--------+
| id|  status|
+---+--------+
|  1|  Active|
|  2|    NULL|
|  3|Inactive|
|  4|    NULL|
+---+--------+

After Filling Null Values:
+---+--------+
| id|  status|
+---+--------+
|  1|  Active|
|  2| Unknown|
|  3|Inactive|
|  4| Unknown|
+---+--------+



# Q6. Count Records for Each City Where Count > 100

This query groups the data by `city`, counts the total number of records for each city,
and displays only those cities whose record count is greater than 100.

In [7]:
# Import required functions
from pyspark.sql.functions import count, col

# Sample Data
data = [
    ("Delhi",),
    ("Delhi",),
    ("Mumbai",),
    ("Delhi",),
    ("Mumbai",),
    ("Chennai",)
]

columns = ["city"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

print("Original Data:")
df.show()

# Group by city and count records
city_count = df.groupBy("city") \
               .agg(count("*").alias("Total_Count"))

# Filter cities where count is greater than 100
result = city_count.filter(col("Total_Count") > 100)

print("Cities with Count Greater Than 100:")
result.show()

Original Data:
+-------+
|   city|
+-------+
|  Delhi|
|  Delhi|
| Mumbai|
|  Delhi|
| Mumbai|
|Chennai|
+-------+

Cities with Count Greater Than 100:
+----+-----------+
|city|Total_Count|
+----+-----------+
+----+-----------+



# Q7. Immutability of Spark DataFrames

Spark DataFrames are **immutable**, which means that once a DataFrame is created,
its data cannot be modified directly.

When performing data cleaning operations such as dropping columns, renaming columns,
or updating values, Spark does not change the original DataFrame. Instead, it creates
and returns a **new DataFrame** with the applied changes.

### Example:

- `df.drop("column_name")` returns a new DataFrame without modifying the original one.
- `df.withColumnRenamed("old_name", "new_name")` creates a new DataFrame with the renamed column.

This approach ensures:
- Data consistency
- Fault tolerance
- Easier parallel processing
- Better optimization by Spark's execution engine

Therefore, developers usually assign the result back to a variable:

```python
df = df.drop("column_name")
df = df.withColumnRenamed("old_name", "new_name")
```


In [8]:
# Original DataFrame remains unchanged unless reassigned

df = df.drop("Bonus")  # Drops the Bonus column and returns a new DataFrame

df = df.withColumnRenamed("Monthly_Salary", "Salary")  # Renames the column

df.show()

+-------+
|   city|
+-------+
|  Delhi|
|  Delhi|
| Mumbai|
|  Delhi|
| Mumbai|
|Chennai|
+-------+



# Q8. Filter Premium Users with Age Between 18 and 30

This query filters the dataset to display only those users:
- Whose age is between **18 and 30 (inclusive)**.
- Whose subscription type is **'Premium'**.

In [9]:
# Import required function
from pyspark.sql.functions import col

# Sample Data
data = [
    ("Rishi", 22, "Premium"),
    ("Aman", 17, "Premium"),
    ("Priya", 25, "Basic"),
    ("Neha", 30, "Premium"),
    ("Karan", 35, "Premium"),
    ("Simran", 28, "Premium")
]

columns = ["name", "age", "subscription"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

print("Original Data:")
df.show()

# Filter users
filtered_df = df.filter(
    (col("age") >= 18) &
    (col("age") <= 30) &
    (col("subscription") == "Premium")
)

print("Filtered Data:")
filtered_df.show()

Original Data:
+------+---+------------+
|  name|age|subscription|
+------+---+------------+
| Rishi| 22|     Premium|
|  Aman| 17|     Premium|
| Priya| 25|       Basic|
|  Neha| 30|     Premium|
| Karan| 35|     Premium|
|Simran| 28|     Premium|
+------+---+------------+

Filtered Data:
+------+---+------------+
|  name|age|subscription|
+------+---+------------+
| Rishi| 22|     Premium|
|  Neha| 30|     Premium|
|Simran| 28|     Premium|
+------+---+------------+



# Q9. Why Should Null Values Be Handled Before Aggregations?

It is important to handle null values before performing mathematical
operations such as `sum()`, `avg()`, `min()`, or `max()` because null values
can lead to incorrect or misleading results.

### Reasons:

1. **Ensures Accurate Calculations**
   - Missing values may reduce the accuracy of averages or totals.

2. **Prevents Unexpected Results**
   - Null values can affect calculations and produce incomplete outputs.

3. **Improves Data Quality**
   - Replacing or removing null values makes the dataset cleaner and more reliable.

4. **Avoids Errors in Analysis**
   - Machine learning models and business reports often require complete data.

### Example:

Before calculating the average salary, null salaries can be replaced with `0`
or another appropriate value using `.na.fill()`.

In [10]:
# Sample Data
data = [
    ("Rishi", 50000),
    ("Aman", None),
    ("Priya", 60000),
    ("Neha", None)
]

columns = ["Name", "Salary"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

print("Original Data:")
df.show()

# Fill null salary values with 0
df_clean = df.na.fill({"Salary": 0})

print("After Handling Null Values:")
df_clean.show()

Original Data:
+-----+------+
| Name|Salary|
+-----+------+
|Rishi| 50000|
| Aman|  NULL|
|Priya| 60000|
| Neha|  NULL|
+-----+------+

After Handling Null Values:
+-----+------+
| Name|Salary|
+-----+------+
|Rishi| 50000|
| Aman|     0|
|Priya| 60000|
| Neha|     0|
+-----+------+



# Q10. Cast `raw_timestamp` to TimestampType and Rename it to `event_time`

This code converts the `raw_timestamp` column to the `TimestampType`
data type and then renames it to `event_time`.

In [11]:
# Import required functions and data type
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType

# Sample Data
data = [
    ("2025-06-21 10:30:00",),
    ("2025-06-22 14:15:45",),
    ("2025-06-23 09:00:10",)
]

columns = ["raw_timestamp"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

print("Original Data:")
df.show()

# Cast raw_timestamp to TimestampType
df = df.withColumn(
    "raw_timestamp",
    col("raw_timestamp").cast(TimestampType())
)

# Rename the column
df = df.withColumnRenamed(
    "raw_timestamp",
    "event_time"
)

print("Updated Data:")
df.show()

# Display Schema
df.printSchema()

Original Data:
+-------------------+
|      raw_timestamp|
+-------------------+
|2025-06-21 10:30:00|
|2025-06-22 14:15:45|
|2025-06-23 09:00:10|
+-------------------+

Updated Data:
+-------------------+
|         event_time|
+-------------------+
|2025-06-21 10:30:00|
|2025-06-22 14:15:45|
|2025-06-23 09:00:10|
+-------------------+

root
 |-- event_time: timestamp (nullable = true)



# Q11. Explain the Shuffle Process in Spark

A **Shuffle** is the process of redistributing data across different partitions
in a Spark cluster so that records with the same key are grouped together.

It commonly occurs during operations such as:
- groupBy()
- reduceByKey()
- join()
- distinct()

### Why does Shuffle happen?

When Spark performs a grouping operation, records with the same key may exist
in different partitions. Spark moves these records across the cluster so that
all matching keys are placed in the same partition.

### Why is it considered a Wide Transformation?

A **wide transformation** requires data movement between partitions.

Unlike narrow transformations (such as `filter()` or `map()`), where each
output partition depends on only one input partition, a shuffle operation
requires communication across multiple partitions.

### Drawbacks of Shuffle

- Increases network traffic
- Requires additional disk I/O
- Consumes more memory
- Slows down execution if not optimized

### Example

The `groupBy("Department")` operation triggers a shuffle because Spark must
bring all records of the same department into the same partition before
performing aggregation.

# Q12. Remove Rows with Null Email or Empty Username

This code identifies and removes rows where:
- The `email` column contains **null values**, OR
- The `username` column is an **empty string ("")**.

The cleaned DataFrame contains only valid records.

In [12]:
# Import required function
from pyspark.sql.functions import col

# Sample Data
data = [
    ("rishi", "rishi@gmail.com"),
    ("aman", None),
    ("", "priya@gmail.com"),
    ("neha", "neha@gmail.com"),
    ("", None)
]

columns = ["username", "email"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

print("Original Data:")
df.show()

# Remove rows where email is null OR username is empty
clean_df = df.filter(
    (col("email").isNotNull()) &
    (col("username") != "")
)

print("Cleaned Data:")
clean_df.show()

Original Data:
+--------+---------------+
|username|          email|
+--------+---------------+
|   rishi|rishi@gmail.com|
|    aman|           NULL|
|        |priya@gmail.com|
|    neha| neha@gmail.com|
|        |           NULL|
+--------+---------------+

Cleaned Data:
+--------+---------------+
|username|          email|
+--------+---------------+
|   rishi|rishi@gmail.com|
|    neha| neha@gmail.com|
+--------+---------------+



# Q13. Calculate Multiple Statistics Using `.agg()`

The `.agg()` function in PySpark is used to perform multiple aggregate
operations on one or more columns in a single query.

In this example, we calculate:

- Minimum Price
- Maximum Price
- Average (Mean) Price

using the `price` column.

In [13]:
# Import required functions
from pyspark.sql.functions import min, max, avg

# Sample Data
data = [
    (100,),
    (250,),
    (300,),
    (450,),
    (500,)
]

columns = ["price"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

print("Original Data:")
df.show()

# Calculate multiple statistics using .agg()
df.agg(
    min("price").alias("Minimum_Price"),
    max("price").alias("Maximum_Price"),
    avg("price").alias("Average_Price")
).show()

Original Data:
+-----+
|price|
+-----+
|  100|
|  250|
|  300|
|  450|
|  500|
+-----+

+-------------+-------------+-------------+
|Minimum_Price|Maximum_Price|Average_Price|
+-------------+-------------+-------------+
|          100|          500|        320.0|
+-------------+-------------+-------------+



# Q14. Risks of Using `inferSchema=true` with Inconsistent Date Formats

When `inferSchema=true` is used, Spark automatically detects the data type
of each column based on the input data.

If the source dataset contains messy or inconsistent date formats, Spark may
infer the wrong data type or fail to parse the values correctly.

## Risks

1. **Incorrect Data Type Detection**
   - Spark may treat a date column as a string instead of a date or timestamp.

2. **Null Values After Parsing**
   - Invalid or inconsistent date formats may be converted to `null`.

3. **Data Quality Issues**
   - Incorrect parsing can lead to inaccurate reports and analysis.

4. **Aggregation and Filtering Errors**
   - Date-based operations such as sorting, filtering, or grouping may produce incorrect results.

5. **Reduced Reliability**
   - Machine learning models and business intelligence reports may be affected by inconsistent data.

## Best Practice

Instead of relying completely on `inferSchema=true`, it is recommended to:

- Define the schema manually whenever possible.
- Clean and standardize date formats before processing.
- Use functions like `to_date()` or `to_timestamp()` with a specified format.

# Q15. Final Data Processing Pipeline

This pipeline performs the following operations:

1. Removes duplicate records.
2. Replaces null values in the `price` column with `0`.
3. Groups the data by `store_id`.
4. Calculates the total revenue for each store using the `sum()` function.

In [14]:
# Import required function
from pyspark.sql.functions import sum

# Sample Data
data = [
    (101, 500),
    (101, None),
    (102, 700),
    (103, 900),
    (102, 700),   # Duplicate
    (104, None)
]

columns = ["store_id", "price"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

print("Original Data:")
df.show()

# Step 1: Remove duplicate rows
df = df.dropDuplicates()

# Step 2: Fill null prices with 0
df = df.na.fill({"price": 0})

# Step 3: Group by store_id and calculate total revenue
result = df.groupBy("store_id") \
           .agg(sum("price").alias("Total_Revenue"))

print("Final Result:")
result.show()

Original Data:
+--------+-----+
|store_id|price|
+--------+-----+
|     101|  500|
|     101| NULL|
|     102|  700|
|     103|  900|
|     102|  700|
|     104| NULL|
+--------+-----+

Final Result:
+--------+-------------+
|store_id|Total_Revenue|
+--------+-------------+
|     103|          900|
|     104|            0|
|     101|          500|
|     102|          700|
+--------+-------------+

